# 📦 RS-LiDAR: Công Cụ Sao Lưu & Đồng Bộ Dữ Liệu Xuyên Tài Khoản Colab

Notebook này được thiết kế chuyên biệt để **chuyển giao toàn bộ Checkpoints, Lookahead Samples, và Kết Quả Thực Nghiệm** giữa các tài khoản Google Colab khác nhau hàng tháng:
- **Không mất mát dữ liệu**: Đóng gói toàn vẹn cấu trúc thư mục `Lookahead_samples`, `Target_samples`, `test_results`.
- **Linh hoạt đa phương thức**:
  1. **Phương pháp 1 (Khuyên dùng - 0 giây)**: Chia sẻ thư mục từ Tài Khoản Cá Nhân sang Colab mới (Không tốn băng thông tải lên/xuống).
  2. **Phương pháp 2**: Nén zip và tải về máy cá nhân / Upload sang Colab mới.
  3. **Phương pháp 3**: Chuyển giao trực tiếp qua Google Drive Link (`gdown`).
- **Tự động khôi phục**: Giải nén chuẩn xác đường dẫn để `LiDAR_Table2_Replication_Colab.ipynb` tiếp tục chạy (`--resume`) ngay lập tức mà không cần sinh lại ảnh.

## 1. Gắn Kết Google Drive Của Tài Khoản Hiện Tại

In [ ]:
import os, sys, shutil, glob, json
from google.colab import drive

# Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = '/content/drive/My Drive' if os.path.exists('/content/drive/My Drive') else '/content/drive/MyDrive'
DRIVE_DIR = f'{base_drive}/RS-LiDAR'

print('✅ Đã gắn kết Drive thành công!')
print('📁 Đường dẫn gốc RS-LiDAR:', DRIVE_DIR)
if not os.path.exists(DRIVE_DIR):
    print('⚠️ Chưa tìm thấy thư mục RS-LiDAR trên Drive. Sẽ tự động khởi tạo khi phục hồi.')
    os.makedirs(DRIVE_DIR, exist_ok=True)
else:
    print('📊 Các thư mục hiện có trên Drive:')
    for item in os.listdir(DRIVE_DIR):
        p = os.path.join(DRIVE_DIR, item)
        if os.path.isdir(p):
            count = len(os.listdir(p))
            print(f'  • {item}/ ({count} mục)')

---
## 🌟 PHẦN A: TÀI KHOẢN CŨ (SAO LƯU & XUẤT DỮ LIỆU)
*Sử dụng phần này khi tài khoản Colab sắp hết hạn để nén và tải dữ liệu về máy cá nhân hoặc lưu trữ.*

In [ ]:
# @title 📦 BƯỚC A1: ĐÓNG GÓI DỮ LIỆU SANG FILE NÉN ZIP
import datetime

# Tùy chọn các thư mục muốn sao lưu
BACKUP_LOOKAHEAD = True  # @param {type:"boolean"} - Chứa latents 50 hạt Phase 1
BACKUP_TARGET = True     # @param {type:"boolean"} - Chứa ảnh mẫu Phase 2 và results.json
BACKUP_TEST_RESULTS = True # @param {type:"boolean"} - Chứa kết quả 5 bài test điểm yếu

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_zip_name = f"RS_LiDAR_Backup_{timestamp}.zip"
backup_zip_path = f"{base_drive}/{backup_zip_name}"

# Thu thập danh sách các thư mục cần nén
items_to_pack = []
if BACKUP_LOOKAHEAD and os.path.exists(f"{DRIVE_DIR}/Lookahead_samples"):
    items_to_pack.append("Lookahead_samples")
if BACKUP_TARGET and os.path.exists(f"{DRIVE_DIR}/Target_samples"):
    items_to_pack.append("Target_samples")
if BACKUP_TEST_RESULTS and os.path.exists(f"{DRIVE_DIR}/test_results"):
    items_to_pack.append("test_results")

if not items_to_pack:
    print("❌ Không tìm thấy thư mục nào để đóng gói!")
else:
    print(f"🚀 Đang nén các thư mục: {items_to_pack}")
    print(f"📁 File zip đầu ra: {backup_zip_path}")
    
    items_str = " ".join(items_to_pack)
    # Sử dụng zip với mức nén tối ưu, hiển thị tiến độ
    !cd "{DRIVE_DIR}" && zip -q -r -1 "{backup_zip_path}" {items_str}
    
    if os.path.exists(backup_zip_path):
        size_gb = os.path.getsize(backup_zip_path) / (1024**3)
        print("=" * 70)
        print(f"✅ ĐÓNG GÓI THÀNH CÔNG!")
        print(f"📦 Tên file: {backup_zip_name}")
        print(f"💾 Dung lượng: {size_gb:.2f} GB")
        print(f"📍 Đã lưu tại: {backup_zip_path}")
        print("=" * 70)
    else:
        print("❌ Có lỗi trong quá trình nén file.")

In [ ]:
# @title ⬇️ BƯỚC A2: TẢI FILE ZIP VỀ MÁY TÍNH CÁ NHÂN (TÙY CHỌN)
# Bạn có thể tải trực tiếp từ giao diện Google Drive trên web, hoặc chạy cell này để tải về qua trình duyệt.
from google.colab import files

DOWNLOAD_TO_LOCAL = False # @param {type:"boolean"} - Đổi thành True nếu muốn tải trực tiếp về máy

if DOWNLOAD_TO_LOCAL and os.path.exists(backup_zip_path):
    print(f"📥 Đang bắt đầu tải {backup_zip_name} về máy tính của bạn...")
    files.download(backup_zip_path)
else:
    print("ℹ️ File zip đã nằm an toàn trên Google Drive của bạn.")
    print("💡 Bạn chỉ cần mở Google Drive trên trình duyệt để tải về máy nếu cần.")

---
## 🌟 PHẦN B: TÀI KHOẢN MỚI (PHỤC HỒI DỮ LIỆU)
*Sử dụng phần này khi bạn đăng nhập vào một tài khoản Colab mới và muốn nạp lại dữ liệu cũ.*

In [ ]:
# @title 📥 BƯỚC B1: CHỌN NGUỒN NẠP DỮ LIỆU
SOURCE_METHOD = "Drive_Zip" # @param ["Drive_Zip", "Google_Drive_Share_Link", "Local_Upload"]
# - Drive_Zip: File zip đã có sẵn trong Google Drive của tài khoản mới (hoặc copy từ Drive cũ sang)
# - Google_Drive_Share_Link: Tải file zip trực tiếp từ link chia sẻ của Google Drive tài khoản cũ (qua gdown)
# - Local_Upload: Tải file zip từ máy tính cá nhân lên Colab

TARGET_ZIP_PATH = ""

if SOURCE_METHOD == "Drive_Zip":
    # Tìm file backup gần nhất trong Drive
    candidates = sorted(glob.glob(f"{base_drive}/RS_LiDAR_Backup_*.zip"))
    if candidates:
        TARGET_ZIP_PATH = candidates[-1]
        print(f"🎯 Đã tự động tìm thấy file backup gần nhất trên Drive: {os.path.basename(TARGET_ZIP_PATH)}")
    else:
        custom_path = input("Nhập tên file hoặc đường dẫn file .zip trên Drive: ")
        TARGET_ZIP_PATH = custom_path if os.path.isabs(custom_path) else f"{base_drive}/{custom_path}"

elif SOURCE_METHOD == "Google_Drive_Share_Link":
    share_link = input("Dán link chia sẻ file zip từ Drive tài khoản cũ (hoặc File ID): ")
    TARGET_ZIP_PATH = f"/content/downloaded_backup.zip"
    print("⬇️ Đang tải file zip từ Google Drive link bằng gdown...")
    !gdown --fuzzy "{share_link}" -O "{TARGET_ZIP_PATH}"

elif SOURCE_METHOD == "Local_Upload":
    from google.colab import files
    print("📤 Hãy chọn file zip từ máy tính để tải lên...")
    uploaded = files.upload()
    if uploaded:
        fname = next(iter(uploaded.keys()))
        TARGET_ZIP_PATH = f"/content/{fname}"
        print(f"✅ Đã tải lên file: {TARGET_ZIP_PATH}")

if TARGET_ZIP_PATH and os.path.exists(TARGET_ZIP_PATH):
    print(f"✅ File nén sẵn sàng để giải nén: {TARGET_ZIP_PATH}")
    print(f"💾 Kích thước: {os.path.getsize(TARGET_ZIP_PATH) / (1024**3):.2f} GB")
else:
    print("❌ Chưa xác định được file zip hợp lệ!")

In [ ]:
# @title 🚀 BƯỚC B2: GIẢI NÉN VÀO ĐÚNG CẤU TRÚC THƯ MỤC RS-LiDAR
if not TARGET_ZIP_PATH or not os.path.exists(TARGET_ZIP_PATH):
    raise FileNotFoundError("❌ Không tìm thấy file backup để giải nén! Hãy hoàn thành Bước B1.")

os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"📦 Đang giải nén {os.path.basename(TARGET_ZIP_PATH)} vào: {DRIVE_DIR}")
print("⏳ Quá trình này có thể mất từ 1 - 3 phút tùy dung lượng dữ liệu...")

# Giải nén trực tiếp vào thư mục RS-LiDAR trên Drive (không ghi đè mù quáng)
!unzip -q -n "{TARGET_ZIP_PATH}" -d "{DRIVE_DIR}"

print("\n" + "=" * 70)
print("✅ PHỤC HỒI DỮ LIỆU HOÀN TẤT TRÊN TÀI KHOẢN COLAB MỚI!")
print("=" * 70)

In [ ]:
# @title 🔍 BƯỚC B3: KIỂM TRA TÍNH TOÀN VẸN & THỐNG KÊ TIẾN ĐỘ THỰC NGHIỆM
import glob

print("📊 BÁO CÁO DỮ LIỆU ĐÃ PHỤC HỒI:")
print("-" * 70)

# 1. Kiểm tra Lookahead samples
look_folders = sorted(glob.glob(f"{DRIVE_DIR}/Lookahead_samples/*"))
if look_folders:
    print(f"📁 [Lookahead_samples] Tìm thấy {len(look_folders)} thư mục:")
    for lf in look_folders:
        p_count = len(glob.glob(f"{lf}/[0-9]*/results.json"))
        lat_count = len(glob.glob(f"{lf}/[0-9]*/samples/latent.pt"))
        print(f"   • {os.path.basename(lf)}: {p_count}/553 prompts đã chấm reward | {lat_count}/553 latents có sẵn")
else:
    print("ℹ️ Chưa có thư mục Lookahead_samples.")

# 2. Kiểm tra Target samples
targ_folders = sorted(glob.glob(f"{DRIVE_DIR}/Target_samples/*"))
if targ_folders:
    print(f"\n📁 [Target_samples] Tìm thấy {len(targ_folders)} thí nghiệm đã sinh ảnh:")
    for tf in targ_folders:
        p_count = len(glob.glob(f"{tf}/[0-9]*/results.json"))
        has_geneval = os.path.exists(f"{tf}/geneval_summary.csv")
        has_pub = os.path.exists(f"{tf}/table2_publication_summary.csv")
        status_str = []
        if has_geneval: status_str.append("GenEval ✅")
        if has_pub: status_str.append("Table 2 ✅")
        extra = f" ({', '.join(status_str)})" if status_str else ""
        print(f"   • {os.path.basename(tf)}: {p_count}/553 prompts hoàn thành{extra}")
else:
    print("ℹ️ Chưa có thư mục Target_samples.")

# 3. Kiểm tra test_results
if os.path.exists(f"{DRIVE_DIR}/test_results"):
    csv_files = glob.glob(f"{DRIVE_DIR}/test_results/*.csv")
    png_files = glob.glob(f"{DRIVE_DIR}/test_results/*.png")
    print(f"\n📁 [test_results] Đã có {len(csv_files)} file CSV và {len(png_files)} biểu đồ kết quả 5 bài test điểm yếu.")

print("-" * 70)
print("🎉 Toàn bộ dữ liệu đã đồng bộ sẵn sàng!")
print("👉 Bây giờ bạn có thể mở 'LiDAR_Table2_Replication_Colab.ipynb', hệ thống sẽ tự động skip các prompt đã hoàn thành nhờ cờ --resume!")

---
## 💡 MẸO VIP: CÁCH CHIA SẺ THƯ MỤC VĨNH VIỄN (KHÔNG TỐN 1 BYTE TẢI LÊN/XUỐNG)

Nếu bạn không muốn mỗi tháng phải nén zip và tải lên/xuống hàng gigabyte dữ liệu, hãy làm theo quy trình sau:

1. **Tạo 1 thư mục gốc trên Tài Khoản Google Drive Cá Nhân (Tài khoản vĩnh viễn)**:
   - Tạo thư mục tên: `RS-LiDAR-Master` trên Drive cá nhân.
2. **Chia sẻ sang tài khoản Colab của tháng mới**:
   - Chuột phải vào `RS-LiDAR-Master` -> Chọn **Chia sẻ (Share)**.
   - Nhập email của tài khoản Colab mới -> Chọn quyền **Người chỉnh sửa (Editor)**.
3. **Thêm lối tắt vào Drive của tài khoản Colab mới**:
   - Đăng nhập vào Drive của tài khoản Colab mới.
   - Vào mục **Được chia sẻ với tôi (Shared with me)**.
   - Chuột phải vào `RS-LiDAR-Master` -> Chọn **Thêm lối tắt vào Drive (Add shortcut to Drive)** -> Đặt tên lối tắt là `RS-LiDAR` ngay tại thư mục gốc của Drive.
4. **Kết quả**:
   - Khi chạy `drive.mount('/content/drive')`, đường dẫn `/content/drive/My Drive/RS-LiDAR` sẽ trỏ thẳng tới dữ liệu trên Drive cá nhân của bạn.
   - Bạn không cần tải lên hay tải xuống bất kỳ file nào; dữ liệu luôn đồng bộ tức thì xuyên suốt mọi tài khoản Colab!